# Knowledge Distillation: 4-Layer Transformer — Alpha & Temperature Grid

This notebook runs 4-Layer-Standard transformer distillation for the same (alpha, temp) configs
that completed for Hybrid-CNN-3Layer, then produces the final comparison.


In [1]:
!pip install transformers datasets accelerate scikit-learn einops

# Configuration & Imports

In [2]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import random
import gc
import os
import json
import time
import sys
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    BertConfig,
    BertForSequenceClassification,
    Trainer,
    TrainingArguments,
    EarlyStoppingCallback,
    TrainerCallback,
)
from datasets import load_dataset
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score

# ===== Configurable Constants =====
TEACHER_NOTEBOOK_SLUG = "ml-project-kaggle15916a635e"
TEACHER_CHECKPOINT_PATH = f"/kaggle/input/{TEACHER_NOTEBOOK_SLUG}/lr_2e-05_results/checkpoint-4375"  # Best: 96.71%
TEACHER_FALLBACK_PATH = f"/kaggle/input/{TEACHER_NOTEBOOK_SLUG}/human_promoter_model"                # Fallback: 96.40%
MODEL_NAME = "quietflamingo/dnabert2-no-flashattention"
MAX_LENGTH = 300
BATCH_SIZE = 16
NUM_EPOCHS = 10
SEED = 42

# ===== Hyperparameter Grid =====
ALPHAS = [0.3, 0.5, 0.7]
TEMPERATURES = [1.0, 3.0, 5.0]

# ===== Reproducibility =====
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# ===== GPU Info =====
device = "cuda" if torch.cuda.is_available() else "cpu"
num_gpus = torch.cuda.device_count()
print(f"Device: {device}")
print(f"Number of GPUs: {num_gpus}")
for i in range(num_gpus):
    name = torch.cuda.get_device_name(i)
    mem = torch.cuda.get_device_properties(i).total_memory / 1e9
    print(f"  GPU {i}: {name} ({mem:.1f} GB)")

print(f"\nHyperparameter grid: {len(ALPHAS)} alphas x {len(TEMPERATURES)} temps = {len(ALPHAS)*len(TEMPERATURES)} combos per architecture")
print(f"Alphas: {ALPHAS}")
print(f"Temperatures: {TEMPERATURES}")

Device: cuda
Number of GPUs: 2
  GPU 0: Tesla T4 (15.6 GB)
  GPU 1: Tesla T4 (15.6 GB)

Hyperparameter grid: 3 alphas x 3 temps = 9 combos per architecture
Alphas: [0.3, 0.5, 0.7]
Temperatures: [1.0, 3.0, 5.0]


# Load Dataset & Tokenize

In [3]:
# Load HUMAN promoter dataset from InstaDeepAI
print("Loading human promoter dataset from InstaDeepAI...")
raw_dataset = load_dataset("InstaDeepAI/nucleotide_transformer_downstream_tasks")

# Filter for promoter_all task only
def filter_promoter_all(example):
    return example["task"] == "promoter_all"

print("Filtering for promoter_all task...")
train_data_raw = raw_dataset["train"].filter(filter_promoter_all)
test_data_raw = raw_dataset["test"].filter(filter_promoter_all)

# Prepare data
def prepare_data(example):
    return {
        "sequence": example["sequence"].upper(),
        "label": int(example["label"]),
    }

train_data = train_data_raw.map(prepare_data)
test_data = test_data_raw.map(prepare_data)

print(f"Training samples: {len(train_data)}, Test samples: {len(test_data)}")

# Tokenize
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)

def tokenize_func(examples):
    return tokenizer(examples["sequence"], padding="max_length", truncation=True, max_length=MAX_LENGTH)

print(f"Tokenizing with max_length={MAX_LENGTH}...")
tokenized_train = train_data.map(tokenize_func, batched=True)
tokenized_test = test_data.map(tokenize_func, batched=True)

# Set format for PyTorch and rename label -> labels for HuggingFace Trainer
tokenized_train.set_format(type="torch", columns=["input_ids", "attention_mask", "label"])
tokenized_test.set_format(type="torch", columns=["input_ids", "attention_mask", "label"])
tokenized_train = tokenized_train.rename_column("label", "labels")
tokenized_test = tokenized_test.rename_column("label", "labels")

train_dataset = tokenized_train
eval_dataset = tokenized_test

print(f"Tokenization complete!")
print(f"Train columns: {train_dataset.column_names}")
print(f"Sample input_ids shape: {train_dataset[0]['input_ids'].shape}")

Loading human promoter dataset from InstaDeepAI...


README.md: 0.00B [00:00, ?B/s]

Resolving data files:   0%|          | 0/18 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/18 [00:00<?, ?it/s]

H3/train.parquet:   0%|          | 0.00/3.48M [00:00<?, ?B/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

Tokenizing with max_length=300...


Map:   0%|          | 0/53276 [00:00<?, ? examples/s]

Map:   0%|          | 0/5920 [00:00<?, ? examples/s]

Tokenization complete!
Train columns: ['sequence', 'name', 'labels', 'task', 'input_ids', 'attention_mask']
Sample input_ids shape: torch.Size([300])


# Load Teacher Model

In [4]:
# Auto-discover teacher model path under /kaggle/input/
# Kaggle mounts notebook outputs at varying depths, e.g.:
#   /kaggle/input/notebooks/username/slug/lr_2e-05_results/checkpoint-4375/
import glob
from safetensors.torch import load_file as load_safetensors
from transformers import AutoConfig
from transformers.dynamic_module_utils import get_class_from_dynamic_module

def find_teacher_model():
    """Search all Kaggle inputs (recursively) for teacher checkpoint files."""
    base = "/kaggle/input"
    
    # Priority 1: Best checkpoint (96.71%)
    for filename in ["model.safetensors", "pytorch_model.bin"]:
        matches = glob.glob(f"{base}/**/lr_2e-05_results/checkpoint-4375/{filename}", recursive=True)
        if matches:
            return os.path.dirname(matches[0]), "best checkpoint (96.71%)"
    
    # Priority 2: Phase 1 model (96.40%)
    for filename in ["model.safetensors", "pytorch_model.bin"]:
        matches = glob.glob(f"{base}/**/human_promoter_model/{filename}", recursive=True)
        if matches:
            return os.path.dirname(matches[0]), "Phase 1 model (96.40%)"
    
    # Priority 3: Any checkpoint with config.json
    matches = glob.glob(f"{base}/**/human_promoter_results/checkpoint-*/config.json", recursive=True)
    if matches:
        return os.path.dirname(matches[0]), "fallback checkpoint"
    
    return None, None

teacher_path, teacher_desc = find_teacher_model()

if teacher_path is None:
    print("Contents of /kaggle/input/:")
    for root, dirs, files in os.walk("/kaggle/input"):
        depth = root.replace("/kaggle/input", "").count(os.sep)
        if depth < 3:
            indent = "  " * depth
            print(f"{indent}{os.path.basename(root)}/")
    raise FileNotFoundError(
        "Teacher model not found!\n"
        "Please add the teacher notebook output as a Kaggle input:\n"
        "  File -> Add Data -> Your Datasets -> Notebook Output"
    )

print(f"Found teacher: {teacher_desc}")
print(f"Path: {teacher_path}")

# Load config from hub and patch missing fields
config = AutoConfig.from_pretrained(MODEL_NAME, trust_remote_code=True)
config.num_labels = 2
config.pad_token_id = tokenizer.pad_token_id  # DNABERT-2 bert_layers.py:39 needs this

# Get DNABERT-2's custom model class from its auto_map
# (bypasses from_pretrained's meta-device context that breaks ALiBi tensor math)
class_ref = config.auto_map["AutoModelForSequenceClassification"]
model_class = get_class_from_dynamic_module(class_ref, MODEL_NAME)

# Direct instantiation on CPU — no meta device wrapping
teacher_model = model_class(config)

# Load fine-tuned checkpoint weights
weights_file = os.path.join(teacher_path, "model.safetensors")
if os.path.exists(weights_file):
    state_dict = load_safetensors(weights_file)
else:
    weights_file = os.path.join(teacher_path, "pytorch_model.bin")
    state_dict = torch.load(weights_file, map_location="cpu", weights_only=True)

teacher_model.load_state_dict(state_dict)
print(f"Loaded fine-tuned weights from: {os.path.basename(weights_file)}")

teacher_model.eval()

teacher_params = sum(p.numel() for p in teacher_model.parameters())
print(f"Teacher model loaded successfully!")
print(f"Teacher parameters: {teacher_params:,} ({teacher_params/1e6:.1f}M)")

Found teacher: best checkpoint (96.71%)
Path: /kaggle/input/notebooks/sheikhrahatmahmud/ml-project-kaggle15916a635e/lr_2e-05_results/checkpoint-4375


bert_layers.py: 0.00B [00:00, ?B/s]

bert_padding.py: 0.00B [00:00, ?B/s]

A new version of the following files was downloaded from https://huggingface.co/quietflamingo/dnabert2-no-flashattention:
- bert_padding.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.


A new version of the following files was downloaded from https://huggingface.co/quietflamingo/dnabert2-no-flashattention:
- bert_layers.py
- bert_padding.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.


/root/.cache/huggingface/modules/transformers_modules/quietflamingo/dnabert2_hyphen_no_hyphen_flashattention/813031b2bf86d9e960a027e2734c908009f31601/bert_layers.py:123: UserWarning: Unable to import Triton; defaulting MosaicBERT attention implementation to pytorch (this will reduce throughput when using this model).
  warnings.warn(


Loaded fine-tuned weights from: model.safetensors
Teacher model loaded successfully!
Teacher parameters: 117,070,082 (117.1M)


# Distillation Trainer & Student Architectures

In [5]:
from transformers.modeling_outputs import SequenceClassifierOutput

# ===== DISTILLATION TRAINER =====
class DistillationTrainer(Trainer):
    def __init__(self, *args, teacher_model=None, alpha=0.5, temperature=3.0, **kwargs):
        super().__init__(*args, **kwargs)
        self.teacher_model = teacher_model.to(self.args.device)
        self.alpha = alpha
        self.temperature = temperature
        self.teacher_model.eval()

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        # Student predictions
        outputs_student = model(**inputs)
        student_logits = outputs_student.logits
        labels = inputs.get("labels")

        # Teacher predictions (no gradients needed)
        with torch.no_grad():
            outputs_teacher = self.teacher_model(**inputs)
            teacher_logits = outputs_teacher.logits

        # Soft Loss: Matching the Teacher's probability distribution
        soft_loss = nn.KLDivLoss(reduction="batchmean")(
            F.log_softmax(student_logits / self.temperature, dim=-1),
            F.softmax(teacher_logits / self.temperature, dim=-1)
        ) * (self.temperature ** 2)

        # Hard Loss: Correctness against ground truth
        hard_loss = F.cross_entropy(student_logits, labels)

        # Final balanced loss
        loss = self.alpha * hard_loss + (1 - self.alpha) * soft_loss

        return (loss, outputs_student) if return_outputs else loss


# ===== HYBRID CNN-TRANSFORMER STUDENT =====
class DNAHybridStudent(nn.Module):
    def __init__(self, layers=2):
        super().__init__()
        # 1. Very small Transformer backbone
        config = BertConfig.from_pretrained("quietflamingo/dnabert2-no-flashattention", num_hidden_layers=layers)
        self.bert = BertForSequenceClassification(config).bert
        
        # 2. CNN Head for Motif Scanning
        self.conv = nn.Conv1d(in_channels=768, out_channels=128, kernel_size=9, padding=4)
        self.pool = nn.AdaptiveMaxPool1d(1)
        self.classifier = nn.Linear(128, 2)

    def forward(self, input_ids, attention_mask=None, labels=None):
        # Transformer pass
        outputs = self.bert(input_ids, attention_mask=attention_mask)
        sequence_output = outputs.last_hidden_state  # [batch, seq_len, 768]
        
        # CNN pass (permute to [batch, 768, seq_len])
        x = sequence_output.permute(0, 2, 1)
        x = torch.relu(self.conv(x))
        x = self.pool(x).squeeze(-1)
        
        logits = self.classifier(x)
        
        loss = None
        if labels is not None:
            loss = nn.CrossEntropyLoss()(logits, labels)
            
        return SequenceClassifierOutput(loss=loss, logits=logits)


# ===== HELPER: PURE TRANSFORMER STUDENT =====
def get_pure_transformer(layers):
    config = BertConfig.from_pretrained(
        "quietflamingo/dnabert2-no-flashattention",
        num_hidden_layers=layers, num_labels=2,
    )
    return BertForSequenceClassification(config)


# ===== COMPUTE METRICS =====
def compute_metrics(pred):
    labels = pred.label_ids
    logits = pred.predictions[0] if isinstance(pred.predictions, tuple) else pred.predictions
    preds = np.argmax(logits, axis=-1)
    return {
        "accuracy": accuracy_score(labels, preds),
        "f1": f1_score(labels, preds),
        "precision": precision_score(labels, preds),
        "recall": recall_score(labels, preds),
    }


# ===== PROGRESS CALLBACK =====
class ProgressCallback(TrainerCallback):
    """Custom callback to force print output in Kaggle"""
    def on_log(self, args, state, control, logs=None, **kwargs):
        if logs:
            print(f"Step {state.global_step}: {logs}", flush=True)

    def on_epoch_begin(self, args, state, control, **kwargs):
        print(f"\n{'='*60}", flush=True)
        print(f"Starting Epoch {int(state.epoch) + 1}/{args.num_train_epochs}", flush=True)
        print(f"{'='*60}", flush=True)

    def on_epoch_end(self, args, state, control, **kwargs):
        print(f"Epoch {int(state.epoch)} completed!", flush=True)


# ===== PARAMETER COUNTING =====
def count_parameters(model):
    return sum(p.numel() for p in model.parameters())


print("All components defined successfully!")

All components defined successfully!


# Run Grid Search: 2 Architectures x Alpha x Temperature

In [6]:
# ===== PREVIOUSLY COMPLETED RESULTS (Hybrid-CNN-3Layer from first run) =====
final_results = {
    "Hybrid-CNN-3Layer_a0.3_t1.0": {
        "architecture": "Hybrid-CNN-3Layer", "alpha": 0.3, "temperature": 1.0,
        "accuracy": 0.9282, "f1": 0.9273, "precision": 0.9393, "recall": 0.9155,
        "loss": 0.1052, "params": 26281346, "compression": 4.454, "train_time_min": 154.2
    },
    "Hybrid-CNN-3Layer_a0.3_t3.0": {
        "architecture": "Hybrid-CNN-3Layer", "alpha": 0.3, "temperature": 3.0,
        "accuracy": 0.9331, "f1": 0.9326, "precision": 0.9402, "recall": 0.9250,
        "loss": 0.1195, "params": 26281346, "compression": 4.454, "train_time_min": 154.3
    },
    "Hybrid-CNN-3Layer_a0.3_t5.0": {
        "architecture": "Hybrid-CNN-3Layer", "alpha": 0.3, "temperature": 5.0,
        "accuracy": 0.9309, "f1": 0.9302, "precision": 0.9400, "recall": 0.9206,
        "loss": 0.1226, "params": 26281346, "compression": 4.454, "train_time_min": 154.3
    },
    "Hybrid-CNN-3Layer_a0.5_t1.0": {
        "architecture": "Hybrid-CNN-3Layer", "alpha": 0.5, "temperature": 1.0,
        "accuracy": 0.9274, "f1": 0.9258, "precision": 0.9461, "recall": 0.9064,
        "loss": 0.1392, "params": 26281346, "compression": 4.454, "train_time_min": 154.1
    },
}

print(f"Loaded {len(final_results)} completed Hybrid-CNN-3Layer results:")
for key, res in final_results.items():
    print(f"  {key}: {res['accuracy']:.2%}")

# ===== MODEL SIZE COMPARISON =====
teacher_params = count_parameters(teacher_model)

print("="*70)
print("MODEL SIZE COMPARISON")
print("="*70)
print(f"{'Model':<25} {'Parameters':>15} {'Layers':>8} {'Compression':>12}")
print("-"*70)
print(f"{'Teacher (DNABERT-2)':<25} {teacher_params:>15,} {'12':>8} {'1.0x':>12}")

_preview_4l = get_pure_transformer(4)
_p4 = count_parameters(_preview_4l)
print(f"{'4-Layer-Standard':<25} {_p4:>15,} {'4':>8} {teacher_params/_p4:>11.1f}x")

_preview_hybrid = DNAHybridStudent(layers=3)
_ph = count_parameters(_preview_hybrid)
print(f"{'Hybrid-CNN-3Layer':<25} {_ph:>15,} {'3+CNN':>8} {teacher_params/_ph:>11.1f}x")
print("="*70)
del _preview_4l, _preview_hybrid
gc.collect()

# ===== RUNS: 4-Layer-Standard for same (alpha, temp) as completed Hybrid =====
runs_to_do = [
    ("4-Layer-Standard", 0.3, 1.0),
    ("4-Layer-Standard", 0.3, 3.0),
    ("4-Layer-Standard", 0.3, 5.0),
    ("4-Layer-Standard", 0.5, 1.0),
]

print(f"\nRuns to complete: {len(runs_to_do)} (4-Layer-Standard)")
print(f"Estimated time: ~{len(runs_to_do) * 154 / 60:.1f} hours")

for run_i, (arch_name, alpha, temperature) in enumerate(runs_to_do):
    run_key = f"{arch_name}_a{alpha}_t{temperature}"

    print(f"\n{'#'*70}")
    print(f"# Run {run_i+1}/{len(runs_to_do)}: {arch_name} | alpha={alpha} | temp={temperature}")
    print(f"{'#'*70}")

    gc.collect()
    torch.cuda.empty_cache()

    random.seed(SEED)
    np.random.seed(SEED)
    torch.manual_seed(SEED)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(SEED)

    student_model = get_pure_transformer(4)
    student_params = count_parameters(student_model)
    print(f"Student parameters: {student_params:,} ({student_params/1e6:.1f}M)")

    args = TrainingArguments(
        output_dir=f"/kaggle/working/{run_key}",
        num_train_epochs=NUM_EPOCHS,
        per_device_train_batch_size=BATCH_SIZE,
        per_device_eval_batch_size=BATCH_SIZE,
        fp16=True,
        logging_steps=100,
        eval_strategy="epoch",
        save_strategy="epoch",
        load_best_model_at_end=True,
        metric_for_best_model="eval_accuracy",
        greater_is_better=True,
        save_total_limit=1,
        report_to="none",
        remove_unused_columns=True,
        warmup_ratio=0.1,
        weight_decay=0.01,
        lr_scheduler_type="cosine",
        learning_rate=2e-5,
    )

    start_time = time.time()

    trainer = DistillationTrainer(
        model=student_model,
        teacher_model=teacher_model,
        alpha=alpha,
        temperature=temperature,
        args=args,
        train_dataset=train_dataset,
        eval_dataset=eval_dataset,
        compute_metrics=compute_metrics,
        callbacks=[
            EarlyStoppingCallback(early_stopping_patience=3),
            ProgressCallback(),
        ],
    )

    print(f"\nStarting training...", flush=True)
    trainer.train()
    train_time = time.time() - start_time

    metrics = trainer.evaluate()
    print(f"\n--- {run_key} Results ---")
    print(f"  Accuracy:  {metrics['eval_accuracy']:.2%}")
    print(f"  F1 Score:  {metrics['eval_f1']:.4f}")
    print(f"  Precision: {metrics['eval_precision']:.4f}")
    print(f"  Recall:    {metrics['eval_recall']:.4f}")
    print(f"  Loss:      {metrics['eval_loss']:.4f}")
    print(f"  Time:      {train_time/60:.1f} min")

    save_dir = f"/kaggle/working/{run_key}_final"
    os.makedirs(save_dir, exist_ok=True)
    student_model.save_pretrained(save_dir)

    final_results[run_key] = {
        "architecture": arch_name,
        "alpha": alpha,
        "temperature": temperature,
        "accuracy": metrics["eval_accuracy"],
        "f1": metrics["eval_f1"],
        "precision": metrics["eval_precision"],
        "recall": metrics["eval_recall"],
        "loss": metrics["eval_loss"],
        "params": student_params,
        "compression": teacher_params / student_params,
        "train_time_min": round(train_time / 60, 1),
    }

    # Save after each run (safe against timeout)
    with open("/kaggle/working/partial_results.json", "w") as f:
        json.dump(final_results, f, indent=2)
    print(f"  Results saved ({len(final_results)} total)")

    del trainer, student_model
    gc.collect()
    torch.cuda.empty_cache()

print(f"\nAll {len(runs_to_do)} runs completed!")
print(f"Total results: {len(final_results)} (4 Hybrid + 4 Transformer)")


Loaded 4 completed Hybrid-CNN-3Layer results:
  Hybrid-CNN-3Layer_a0.3_t1.0: 92.82%
  Hybrid-CNN-3Layer_a0.3_t3.0: 93.31%
  Hybrid-CNN-3Layer_a0.3_t5.0: 93.09%
  Hybrid-CNN-3Layer_a0.5_t1.0: 92.74%
MODEL SIZE COMPARISON
Model                          Parameters   Layers  Compression
----------------------------------------------------------------------
Teacher (DNABERT-2)           117,070,082       12         1.0x


4-Layer-Standard               32,485,634        4         3.6x


Hybrid-CNN-3Layer              26,281,346    3+CNN         4.5x



Runs to complete: 4 (4-Layer-Standard)
Estimated time: ~10.3 hours

######################################################################
# Run 1/4: 4-Layer-Standard | alpha=0.3 | temp=1.0
######################################################################


Student parameters: 32,485,634 (32.5M)


...

Step 16000: {'loss': 0.09022007942199707, 'grad_norm': 3.289018154144287, 'learning_rate': 9.299165686633471e-08, 'epoch': 9.60960960960961}


Step 16100: {'loss': 0.08908653259277344, 'grad_norm': 5.43095588684082, 'learning_rate': 6.664636386630286e-08, 'epoch': 9.66966966966967}


Step 16200: {'loss': 0.09188619613647461, 'grad_norm': 4.740228176116943, 'learning_rate': 4.466689503498045e-08, 'epoch': 9.72972972972973}


Step 16300: {'loss': 0.08053483009338379, 'grad_norm': 7.39125919342041, 'learning_rate': 2.7062910603921078e-08, 'epoch': 9.78978978978979}


Step 16400: {'loss': 0.08986308097839356, 'grad_norm': 8.240013122558594, 'learning_rate': 1.3842147728522215e-08, 'epoch': 9.84984984984985}


# Results Summary & Visualization

In [ ]:
import matplotlib.pyplot as plt
import matplotlib
import pandas as pd

TEACHER_ACCURACY = 0.9671

# ===== FULL RESULTS TABLE =====
print("="*110)
print("KNOWLEDGE DISTILLATION: Hybrid-CNN-3Layer vs 4-Layer-Standard")
print("="*110)
print(f"{'Run':<35} {'Alpha':>6} {'Temp':>6} {'Accuracy':>10} {'F1':>8} {'Params':>10} {'Compress':>10} {'Time':>8}")
print("-"*110)
print(f"{'Teacher (DNABERT-2)':<35} {'--':>6} {'--':>6} {TEACHER_ACCURACY:>9.2%} {'--':>8} {teacher_params/1e6:>8.1f}M {'1.0x':>10} {'--':>8}")
print("-"*110)

for key, res in sorted(final_results.items()):
    print(f"{key:<35} {res['alpha']:>6.1f} {res['temperature']:>6.1f} {res['accuracy']:>9.2%} {res['f1']:>8.4f} {res['params']/1e6:>8.1f}M {res['compression']:>9.1f}x {res['train_time_min']:>6.1f}m")
print("="*110)

# ===== HEAD-TO-HEAD COMPARISON =====
configs = [(0.3, 1.0), (0.3, 3.0), (0.3, 5.0), (0.5, 1.0)]
arch_names = ["Hybrid-CNN-3Layer", "4-Layer-Standard"]

print(f"\n{'='*80}")
print("HEAD-TO-HEAD: Same (alpha, temp) — Hybrid vs 4-Layer Transformer")
print(f"{'='*80}")
print(f"{'Alpha':>6} {'Temp':>6} {'Hybrid Acc':>12} {'4L-Trans Acc':>14} {'Winner':>20}")
print("-"*80)

for alpha, temp in configs:
    h_key = f"Hybrid-CNN-3Layer_a{alpha}_t{temp}"
    t_key = f"4-Layer-Standard_a{alpha}_t{temp}"
    h_acc = final_results.get(h_key, {}).get("accuracy", 0)
    t_acc = final_results.get(t_key, {}).get("accuracy", 0)
    diff = h_acc - t_acc
    winner = "Hybrid" if diff > 0 else "4L-Transformer" if diff < 0 else "Tie"
    print(f"{alpha:>6.1f} {temp:>6.1f} {h_acc:>11.2%} {t_acc:>13.2%} {winner:>15} ({abs(diff):.2%})")
print("="*80)

# ===== BEST CONFIG PER ARCHITECTURE =====
print(f"\n{'='*70}")
print("BEST CONFIGURATION PER ARCHITECTURE")
print(f"{'='*70}")
for arch in arch_names:
    arch_results = {k: v for k, v in final_results.items() if v["architecture"] == arch}
    if not arch_results:
        continue
    best_key = max(arch_results, key=lambda k: arch_results[k]["accuracy"])
    best = arch_results[best_key]
    retention = best["accuracy"] / TEACHER_ACCURACY * 100
    print(f"\n  {arch}:")
    print(f"    Best config: alpha={best['alpha']}, temp={best['temperature']}")
    print(f"    Accuracy: {best['accuracy']:.2%} ({retention:.1f}% of teacher)")
    print(f"    F1: {best['f1']:.4f}")
    print(f"    Compression: {best['compression']:.1f}x")

# ===== GROUPED BAR CHART: Side-by-side comparison =====
fig, axes = plt.subplots(1, 3, figsize=(20, 6))

config_labels = [f"a={a},T={t}" for a, t in configs]
hybrid_accs = [final_results.get(f"Hybrid-CNN-3Layer_a{a}_t{t}", {}).get("accuracy", 0) for a, t in configs]
trans_accs = [final_results.get(f"4-Layer-Standard_a{a}_t{t}", {}).get("accuracy", 0) for a, t in configs]

x = np.arange(len(configs))
width = 0.35

# 1. Accuracy comparison (grouped bars)
bars1 = axes[0].bar(x - width/2, hybrid_accs, width, label="Hybrid-CNN-3Layer", color="#2ecc71")
bars2 = axes[0].bar(x + width/2, trans_accs, width, label="4-Layer-Standard", color="#3498db")
axes[0].axhline(y=TEACHER_ACCURACY, color="gold", linestyle="--", linewidth=2, label=f"Teacher ({TEACHER_ACCURACY:.2%})")
axes[0].set_ylabel("Accuracy")
axes[0].set_title("Accuracy by (Alpha, Temp) Config")
axes[0].set_xticks(x)
axes[0].set_xticklabels(config_labels, rotation=15)
axes[0].legend(fontsize=8)
axes[0].set_ylim(0.88, 0.97)
for bar in bars1:
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.001, f"{bar.get_height():.2%}", ha="center", fontsize=7)
for bar in bars2:
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.001, f"{bar.get_height():.2%}", ha="center", fontsize=7)

# 2. Model size comparison
best_per_arch = {}
for arch in arch_names:
    arch_results = {k: v for k, v in final_results.items() if v["architecture"] == arch}
    if arch_results:
        best_key = max(arch_results, key=lambda k: arch_results[k]["accuracy"])
        best_per_arch[arch] = arch_results[best_key]

bar_names = ["Teacher"] + list(best_per_arch.keys())
bar_params = [teacher_params / 1e6] + [best_per_arch[n]["params"] / 1e6 for n in best_per_arch]
bar_colors = ["gold", "#2ecc71", "#3498db"]
bars3 = axes[1].bar(bar_names, bar_params, color=bar_colors[:len(bar_names)])
axes[1].set_ylabel("Parameters (M)")
axes[1].set_title("Model Size Comparison")
for bar, p in zip(bars3, bar_params):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1, f"{p:.1f}M", ha="center", fontsize=9)

# 3. Best accuracy per architecture
best_names = list(best_per_arch.keys())
best_accs = [best_per_arch[n]["accuracy"] for n in best_names]
bars4 = axes[2].bar(best_names, best_accs, color=["#2ecc71", "#3498db"][:len(best_names)])
axes[2].axhline(y=TEACHER_ACCURACY, color="gold", linestyle="--", linewidth=2, label=f"Teacher ({TEACHER_ACCURACY:.2%})")
axes[2].set_ylabel("Accuracy")
axes[2].set_title("Best Accuracy per Architecture")
axes[2].legend()
axes[2].set_ylim(0.88, 0.97)
for bar, acc in zip(bars4, best_accs):
    axes[2].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.002, f"{acc:.2%}", ha="center", fontsize=9)

plt.tight_layout()
plt.savefig("/kaggle/working/distillation_comparison.png", dpi=150, bbox_inches="tight")
plt.show()
print("Charts saved to /kaggle/working/distillation_comparison.png")

# ===== SAVE RESULTS JSON =====
results_json = {
    "teacher_accuracy": TEACHER_ACCURACY,
    "teacher_params": teacher_params,
    "configs_tested": [{"alpha": a, "temperature": t} for a, t in configs],
    "runs": final_results,
}
with open("/kaggle/working/distillation_results.json", "w") as f:
    json.dump(results_json, f, indent=2)
print("Results saved to /kaggle/working/distillation_results.json")
